# 14 · Neural-network foundations lab

Unit I building blocks, run on toy data you can check by hand, then one real-data MLP.

**Evidence types in this notebook:** sections 1–5 use *synthetic toy inputs* (teaching demos, not accuracy). Section 6 reads the saved result of the sealed real-data study `MLP-CONTROLLED-*` (Birmingham TRAIN/VALIDATION only; TEST never read).

In [1]:
import sys, json, glob
from pathlib import Path
import numpy as np
ROOT = Path.cwd().resolve()
while not (ROOT/'src/smart_parking').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from smart_parking.neural_basics import gate, perceptron_train, perceptron_predict, sigmoid, sigmoid_derivative
from smart_parking.learning_rules import hebbian, oja, competitive, delta_rule, hopfield_store, hopfield_recall
from smart_parking.mlp import MLP
X2 = np.array([[0,0],[0,1],[1,0],[1,1]])

## 1 · McCulloch–Pitts neuron

$y = [\,w\cdot x \ge \theta\,]$ — binary inputs, fixed weights, a threshold. No learning: the weights are set by hand. AND uses $w=(1,1),\ \theta=2$; OR uses $\theta=1$.

In [2]:
for g in ['AND','OR','NAND']:
    print(g, gate(g, X2))
print('NOT', gate('NOT', np.array([[0],[1]])))

AND [0 0 0 1]
OR [0 1 1 1]
NAND [1 1 1 0]
NOT [1 0]


## 2 · Perceptron learning rule

$w \leftarrow w + \eta\,(t-\hat y)\,x$. It reaches zero errors on linearly separable data (AND) but **never** on XOR — one straight line cannot separate XOR, which is why we need hidden layers.

In [3]:
w,b,h = perceptron_train(X2,[0,0,0,1]); print('AND errors per epoch:', h, '-> predictions', perceptron_predict(X2,w,b))
_,_,hx = perceptron_train(X2,[0,1,1,0],epochs=20); print('XOR errors per epoch (never 0):', hx)

AND errors per epoch: [2, 2, 1, 1, 1, 3, 0] -> predictions [0 0 0 1]
XOR errors per epoch (never 0): [3, 2, 3, 4, 3, 4, 4, 3, 2, 4, 1, 4, 4, 2, 3, 2, 2, 3, 2, 2]


## 3 · Sigmoid neuron and its derivative

$\sigma(z)=1/(1+e^{-z})$, $\sigma'(z)=\sigma(z)(1-\sigma(z))$. Checked against a central finite difference.

In [4]:
z=np.linspace(-4,4,5); e=1e-6
print(np.round(sigmoid(z),4)); print('max |analytic - numeric| =', np.abs(sigmoid_derivative(z)-(sigmoid(z+e)-sigmoid(z-e))/(2*e)).max())

[0.018  0.1192 0.5    0.8808 0.982 ]
max |analytic - numeric| = 3.494449174468173e-11


## 4 · Tensors: shapes, indexing, broadcasting, reshape, products

A batch of 2 RGB images 4×3 is a tensor of shape (N, H, W, C).

In [5]:
x = np.arange(2*4*3*3).reshape(2,4,3,3)          # N,H,W,C
print('shape', x.shape, '| first image, row 0, all columns, red:', x[0,0,:,0])
mean = x.mean(axis=(0,1,2)); print('per-channel mean (broadcast over N,H,W):', mean)
print('subtract mean broadcasts (3,) over (2,4,3,3):', (x-mean).shape)
chw = x.transpose(0,3,1,2); print('NHWC -> NCHW', chw.shape, '| flatten each image:', x.reshape(2,-1).shape)
A=np.array([[1,2],[3,4]]); B=np.array([[0,1],[1,0]])
print('elementwise A*B=\n',A*B,'\nmatrix A@B=\n',A@B)

shape (2, 4, 3, 3) | first image, row 0, all columns, red: [0 3 6]
per-channel mean (broadcast over N,H,W): [34.5 35.5 36.5]
subtract mean broadcasts (3,) over (2,4,3,3): (2, 4, 3, 3)
NHWC -> NCHW (2, 3, 4, 3) | flatten each image: (2, 36)
elementwise A*B=
 [[0 2]
 [3 0]] 
matrix A@B=
 [[2 1]
 [4 3]]


## 5 · Learning rules compared

| Rule | Update | Uses labels? |
|---|---|---|
| Hebb | $\Delta w=\eta\,y\,x$ | no |
| Oja | $\Delta w=\eta\,y\,(x-y\,w)$ | no |
| Competitive | winner moves: $\Delta w_k=\eta(x-w_k)$ | no |
| Delta / LMS | $\Delta w=\eta\,(t-y)\,x$ | yes |
| Hopfield | store $W=\tfrac1N\sum pp^\top$, recall lowers $E=-\tfrac12 s^\top W s$ | patterns |

In [6]:
rng=np.random.default_rng(0); x=rng.normal(0,1,(300,2))@np.array([[3.,0.],[0.,.5]])
_,hn=hebbian(x,epochs=3); print('Hebb |w| grows:', np.round(hn,2))
w,on=oja(x,epochs=30,learning_rate=.005); print('Oja |w| ->', round(on[-1],3), 'direction', np.round(w,3), '(first principal axis)')
c=np.concatenate([rng.normal(m,.1,(30,2)) for m in ([0,0],[3,3])]); _,a=competitive(c,units=2); print('competitive clusters:', np.bincount(a))
_,_,mse=delta_rule(x, x@np.array([2.,-1.])+.5, epochs=10); print('delta-rule MSE per epoch:', np.round(mse,4))
p=np.array([[1,-1,1,-1,1,-1,1,-1],[1,1,1,1,-1,-1,-1,-1]]); W=hopfield_store(p); noisy=p[0].copy(); noisy[0]*=-1
s,E=hopfield_recall(W,noisy); print('Hopfield recalled stored pattern:', bool((s==p[0]).all()), '| energy start->end', E[0], '->', E[-1])

Hebb |w| grows: [4.49806793e+08 1.59924244e+19 5.66155985e+29]
Oja |w| -> 1.0 direction [1.    0.022] (first principal axis)
competitive clusters: [30 30]
delta-rule MSE per epoch: [0.0496 0.011  0.0024 0.0006 0.0001 0.     0.     0.     0.     0.    ]
Hopfield recalled stored pattern: True | energy start->end -1.5 -> -3.0


## 6 · Optimisers and a real-data MLP

Plain momentum: $v\leftarrow\mu v-\eta g,\ p\leftarrow p+v$. Nesterov looks ahead: $p\leftarrow p+\mu v-\eta g$. Two steps with constant gradient 1, $\eta=0.1,\ \mu=0.5$ give −0.25 and −0.325.

In [7]:
for nest in (False,True):
    m=MLP(); w0=m.parameters['b2'].copy(); g={k:np.ones_like(v) for k,v in m.parameters.items()}
    m.step(g,.1,.5,nesterov=nest); m.step(g,.1,.5,nesterov=nest); print('nesterov' if nest else 'momentum', m.parameters['b2']-w0)

momentum [-0.25]
nesterov [-0.325]


The sealed study changed **one factor at a time** from a baseline, with 3 seeds each, on real Birmingham car-park windows (will the next reading within 45 min be ≥85% full?). A change counts as real only if it beats 2× the baseline's seed spread.

In [8]:
run = json.loads(sorted(ROOT.glob('experiments/results/MLP-CONTROLLED-*/run.json'))[-1].read_text())
print('train/validation windows:', run['train_windows'], run['validation_windows'], '| persistence rule balanced acc:', round(run['persistence_reference_balanced_accuracy'],4))
for n,r in run['results'].items():
    e=r.get('effect',{})
    print(f"{n:20s} val loss {r['validation_loss_mean']:.4f}±{r['validation_loss_sd']:.4f}  bal.acc {r['balanced_accuracy_mean']:.4f}±{r['balanced_accuracy_sd']:.4f}  real effect: {e.get('balanced_accuracy_effect_real','-')}")

train/validation windows: 23126 4469 | persistence rule balanced acc: 0.9236
baseline             val loss 0.0489±0.0003  bal.acc 0.9748±0.0023  real effect: -
optimizer=sgd        val loss 0.0517±0.0003  bal.acc 0.9693±0.0004  real effect: True
optimizer=nesterov   val loss 0.0488±0.0006  bal.acc 0.9741±0.0031  real effect: False
learning_rate=0.01   val loss 0.0490±0.0003  bal.acc 0.9709±0.0020  real effect: False
learning_rate=0.2    val loss 0.0506±0.0009  bal.acc 0.9741±0.0014  real effect: False
init_scale=0.1       val loss 0.0489±0.0010  bal.acc 0.9733±0.0027  real effect: False
init_scale=3         val loss 0.0509±0.0014  bal.acc 0.9719±0.0028  real effect: False
normalization=off    val loss 0.0716±0.0042  bal.acc 0.9459±0.0189  real effect: True
l2=1e-3              val loss 0.0554±0.0013  bal.acc 0.9682±0.0055  real effect: True
dropout=0.2          val loss 0.0512±0.0017  bal.acc 0.9640±0.0027  real effect: True
batch_size=16        val loss 0.0524±0.0006  bal.acc 0.9729±0

**Read it carefully:** the MLP beats the simple persistence rule on these validation days, but this is a course study on reused validation data, not deployment accuracy. Normalisation mattered most; Nesterov, a smaller learning rate and batch 256 were within seed noise here.